# Retail Sales Prediction Using Machine Learning

## Project Objective
Predict daily retail store sales using historical sales, promotion, holiday, store, assortment, and competition information.

### Dataset
- Sales data: daily store-level observations
- Store data: store characteristics
- Target: `Sales`

### Machine Learning Algorithms
1. Random Forest Regressor
2. HistGradientBoosting Regressor

### Important forecasting decision
`Customers` is not used as a model feature because future customer counts would generally not be available when making a future sales forecast. This avoids using information that may not exist at prediction time.


## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")


## 2. Load the Two CSV Files

Keep both CSV files in the same folder as this notebook, or change the paths below.


In [ ]:
SALES_PATH = "Salesdata-DS DATA.csv"
STORE_PATH = "store-DS DATA.csv"

sales = pd.read_csv(SALES_PATH, low_memory=False)
store = pd.read_csv(STORE_PATH)

print("Sales shape:", sales.shape)
print("Store shape:", store.shape)


## 3. Understand the Data

In [ ]:
print("Sales columns:")
print(sales.columns.tolist())

print("\nStore columns:")
print(store.columns.tolist())

print("\nSales data types:")
display(sales.dtypes)

print("\nStore data types:")
display(store.dtypes)

print("\nSales sample:")
display(sales.head())

print("\nStore sample:")
display(store.head())


In [ ]:
print("Sales missing values:")
display(sales.isnull().sum().sort_values(ascending=False))

print("\nStore missing values:")
display(store.isnull().sum().sort_values(ascending=False))

print("\nDuplicate sales rows:", sales.duplicated().sum())
print("Duplicate store rows:", store.duplicated().sum())


## 4. Data Cleaning and Dataset Merge

The `Store` column is the common key between the daily sales data and store information.


In [ ]:
# Convert date
sales["Date"] = pd.to_datetime(sales["Date"], dayfirst=True, errors="coerce")

# Remove duplicate rows if any
sales = sales.drop_duplicates().copy()
store = store.drop_duplicates(subset=["Store"]).copy()

# Convert relevant holiday columns to strings for consistent categorical handling
sales["StateHoliday"] = sales["StateHoliday"].astype(str)

# Merge daily sales with store-level information
df = sales.merge(store, on="Store", how="left")

print("Merged dataset shape:", df.shape)
display(df.head())


In [ ]:
# Missing-value summary after merge
missing = df.isnull().sum().sort_values(ascending=False)
display(missing[missing > 0])

# Fill store attributes where missing means the information is unavailable/not applicable
for col in ["CompetitionOpenSinceMonth", "CompetitionOpenSinceYear",
            "Promo2SinceWeek", "Promo2SinceYear"]:
    df[col] = df[col].fillna(0)

df["CompetitionDistance"] = df["CompetitionDistance"].fillna(
    df["CompetitionDistance"].median()
)

df["PromoInterval"] = df["PromoInterval"].fillna("None")

print("Remaining missing values:", int(df.isnull().sum().sum()))


## 5. Exploratory Data Analysis (EDA)

We examine sales by day, promotion, store type, and month.


In [ ]:
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
print("Number of stores:", df["Store"].nunique())
print("Open/closed counts:")
display(df["Open"].value_counts())

print("\nSales summary:")
display(df["Sales"].describe())


In [ ]:
# Average sales by day of week
dow_sales = df.groupby("DayOfWeek")["Sales"].mean()

plt.figure(figsize=(8, 4))
dow_sales.plot(kind="bar")
plt.title("Average Sales by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Average Sales")
plt.tight_layout()
plt.show()


In [ ]:
# Average sales with and without promotion
promo_sales = df.groupby("Promo")["Sales"].mean()

plt.figure(figsize=(6, 4))
promo_sales.plot(kind="bar")
plt.title("Average Sales by Promotion")
plt.xlabel("Promo")
plt.ylabel("Average Sales")
plt.tight_layout()
plt.show()


In [ ]:
# Average sales by store type
store_type_sales = df.groupby("StoreType")["Sales"].mean().sort_values(ascending=False)

plt.figure(figsize=(7, 4))
store_type_sales.plot(kind="bar")
plt.title("Average Sales by Store Type")
plt.xlabel("Store Type")
plt.ylabel("Average Sales")
plt.tight_layout()
plt.show()


In [ ]:
# Monthly sales trend
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month

monthly_sales = df.groupby(["Year", "Month"])["Sales"].mean().reset_index()
monthly_sales["Period"] = pd.to_datetime(
    monthly_sales["Year"].astype(str) + "-" +
    monthly_sales["Month"].astype(str) + "-01"
)

plt.figure(figsize=(12, 4))
plt.plot(monthly_sales["Period"], monthly_sales["Sales"])
plt.title("Average Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Average Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 6. Feature Engineering

Create calendar features from the date. We also calculate store age relative to the competition opening information where possible.

`Customers` is deliberately excluded from the predictive feature set because it is not a reliable future-known variable.


In [ ]:
df["Day"] = df["Date"].dt.day
df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
df["Quarter"] = df["Date"].dt.quarter
df["IsWeekend"] = (df["Date"].dt.dayofweek >= 5).astype(int)

# A simple competition-age feature
df["CompetitionAgeMonths"] = (
    (df["Year"] - df["CompetitionOpenSinceYear"]) * 12
    + (df["Month"] - df["CompetitionOpenSinceMonth"])
)

df["CompetitionAgeMonths"] = df["CompetitionAgeMonths"].clip(lower=0)

print("Feature engineering completed.")
display(df.head())


## 7. Prepare the Modeling Dataset

We model only records where the store was open because closed stores have zero sales and are not useful for learning normal sales demand.

A chronological validation split is used:
- Earlier observations → training
- Most recent observations → validation

This is more appropriate for forecasting than randomly mixing future observations into training.


In [ ]:
model_df = df[df["Open"] == 1].copy()
model_df = model_df.sort_values("Date").reset_index(drop=True)

# Use the last 6 weeks of available open-store observations as validation
cutoff_date = model_df["Date"].max() - pd.Timedelta(weeks=6)

train_df = model_df[model_df["Date"] < cutoff_date].copy()
valid_df = model_df[model_df["Date"] >= cutoff_date].copy()

print("Training rows:", len(train_df))
print("Validation rows:", len(valid_df))
print("Training date range:", train_df["Date"].min().date(), "to", train_df["Date"].max().date())
print("Validation date range:", valid_df["Date"].min().date(), "to", valid_df["Date"].max().date())


## 8. Select Features

Categorical variables are one-hot encoded.

Numerical variables are passed directly to the models.


In [ ]:
target = "Sales"

features = [
    "Store",
    "DayOfWeek",
    "Year",
    "Month",
    "Day",
    "WeekOfYear",
    "Quarter",
    "IsWeekend",
    "Promo",
    "Promo2",
    "StateHoliday",
    "SchoolHoliday",
    "StoreType",
    "Assortment",
    "CompetitionDistance",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",
    "Promo2SinceWeek",
    "Promo2SinceYear",
    "CompetitionAgeMonths"
]

X_train = train_df[features]
y_train = train_df[target]

X_valid = valid_df[features]
y_valid = valid_df[target]

categorical_features = [
    "StateHoliday",
    "StoreType",
    "Assortment"
]

numeric_features = [
    col for col in features if col not in categorical_features
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

print("Number of input features before encoding:", len(features))


## 9. Model 1 — Random Forest Regressor

In [ ]:
# For faster training on very large data, use a reproducible sample.
# The validation set remains untouched.
MAX_TRAIN_ROWS = 200_000

if len(X_train) > MAX_TRAIN_ROWS:
    sample_idx = X_train.sample(
        n=MAX_TRAIN_ROWS,
        random_state=42
    ).index
    X_train_model = X_train.loc[sample_idx]
    y_train_model = y_train.loc[sample_idx]
else:
    X_train_model = X_train
    y_train_model = y_train

rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=80,
        max_depth=20,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train_model, y_train_model)

rf_pred = rf_model.predict(X_valid)

print("Random Forest training completed.")


## 10. Model 2 — HistGradientBoosting Regressor

In [ ]:
# HistGradientBoosting works with numeric arrays, so we transform the data first.
X_train_encoded = preprocessor.fit_transform(X_train_model)
X_valid_encoded = preprocessor.transform(X_valid)

hgb_model = HistGradientBoostingRegressor(
    max_iter=250,
    learning_rate=0.08,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

hgb_model.fit(X_train_encoded, y_train_model)

hgb_pred = hgb_model.predict(X_valid_encoded)

print("HistGradientBoosting training completed.")


## 11. Model Evaluation

Metrics:
- **MAE**: average absolute prediction error; lower is better.
- **RMSE**: penalizes larger errors more strongly; lower is better.
- **R²**: proportion of target variation explained by the model; higher is better.


In [ ]:
def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    return {
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

results = pd.DataFrame([
    evaluate_model("Random Forest", y_valid, rf_pred),
    evaluate_model("HistGradientBoosting", y_valid, hgb_pred)
])

display(results.sort_values("RMSE"))


## 12. Compare Actual vs Predicted Sales

In [ ]:
# Select the better model using validation RMSE
best_model_name = results.sort_values("RMSE").iloc[0]["Model"]
print("Best model based on RMSE:", best_model_name)

best_pred = rf_pred if best_model_name == "Random Forest" else hgb_pred

comparison = valid_df[["Store", "Date", "Sales"]].copy()
comparison["PredictedSales"] = best_pred
comparison["AbsoluteError"] = abs(comparison["Sales"] - comparison["PredictedSales"])

display(comparison.head(20))


In [ ]:
# Plot a sample of actual vs predicted values
plot_df = comparison.sample(
    n=min(3000, len(comparison)),
    random_state=42
).sort_values("Date")

plt.figure(figsize=(12, 5))
plt.plot(plot_df["Date"], plot_df["Sales"], label="Actual", alpha=0.7)
plt.plot(plot_df["Date"], plot_df["PredictedSales"], label="Predicted", alpha=0.7)
plt.title(f"Actual vs Predicted Sales - {best_model_name}")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 13. Store-wise Batch Prediction

The project requires batch prediction by store ID. This section produces predictions for the validation period and summarizes them by store.


In [ ]:
store_predictions = comparison[
    ["Store", "Date", "PredictedSales"]
].sort_values(["Store", "Date"])

print("Store-wise prediction sample:")
display(store_predictions.head(30))

store_summary = (
    store_predictions
    .groupby("Store")["PredictedSales"]
    .agg(["count", "mean", "sum"])
    .reset_index()
    .rename(columns={
        "count": "PredictedDays",
        "mean": "AveragePredictedDailySales",
        "sum": "TotalPredictedSales"
    })
)

display(store_summary.head(20))


## 14. Save Prediction Output

In [ ]:
store_predictions.to_csv(
    "store_wise_sales_predictions.csv",
    index=False
)

results.to_csv(
    "model_comparison.csv",
    index=False
)

print("Saved:")
print("- store_wise_sales_predictions.csv")
print("- model_comparison.csv")


## 15. Business Insights

Use the EDA and model results to explain:

1. Which days have higher/lower average sales.
2. Whether promotions are associated with higher sales.
3. Which store types have stronger average sales.
4. How sales vary over months and seasons.
5. How competition distance relates to store performance.
6. Which ML model performs better on the recent validation period.
7. Why store-wise prediction can help managers plan inventory, staffing, and promotions.

### Final Conclusion
The project combines historical retail sales with store-level information to build a data-driven sales forecasting solution. Two machine learning algorithms are compared using a chronological validation period, and the selected model is used to generate store-wise batch predictions.


# Interview Explanation

**"My project is Retail Sales Prediction using Machine Learning. I had two datasets: daily sales data and store information. I first explored and cleaned the data, converted the date into useful calendar features, and merged both datasets using Store ID. I excluded Customers from the final predictive features because future customer counts may not be available during forecasting. I then trained two regression algorithms, Random Forest and HistGradientBoosting, using a chronological train-validation split. I compared them using MAE, RMSE and R², selected the better model based on validation RMSE, and generated store-wise batch sales predictions."**
